In [ ]:
import duckdb
import toml
from pathlib import Path

In [ ]:
def load_secrets(secrets_file="../.secrets.toml"):
    try:
        secrets = toml.load(secrets_file)
        return secrets
    except Exception as e:
        print(f"Failed to load secrets: {e}")
        return None

In [ ]:
secrets = load_secrets()

In [ ]:
duckdb.__version__

In [ ]:
con_md = duckdb.connect(f'md:my_db?motherduck_token={secrets["MD_TOKEN"]}')

In [ ]:
# InvalidInputException: Invalid Input Error: Initialization function "motherduck_init" from file "/Users/mjboothaus/.duckdb/extensions/v0.10.2/osx_arm64/motherduck.duckdb_extension" threw an exception: "Initialization function "motherduck_init" from file "/Users/mjboothaus/.duckdb/extensions/v0.10.2/osx_arm64/motherduck.v0.10.2-2024-05-282.duckdb_extension" threw an exception: "Request failed:  (UNAVAILABLE, RPC 'GET_WELCOME_PACK', request id: '818fa3cd-bf2a-43d0-8505-6ceee1a94ba5')""

In [ ]:
def copy_local_ddb_to_motherduck(local_ddb, db_name=None, dry_run=True):
    if db_name is None:
        db_name = Path(local_ddb).name
    local_con = duckdb.connect(local_ddb)
    if not dry_run:
        local_con.sql("LOAD motherduck")
        local_con.sql(f"CREATE DATABASE {db_name} FROM CURRENT_DATABASE()") 
        print(f'Created database {db_name} on https://motherduck.com')
    local_con.close()
    return db_name

In [ ]:
db_name = copy_local_ddb_to_motherduck("../data/experiments_October2023.ddb")

In [ ]:
db_name

In [ ]:
con_md.sql("SHOW ALL DATABASES;")

In [ ]:
con_md.sql("SHOW ALL TABLES")

In [ ]:
con_md.close()

In [ ]:

con_md_share = duckdb.connect()

In [ ]:
con_md_share.sql("ATTACH 'md:_share/clouddb/cbe96e98-c6b3-4e13-86b4-c3b42cf95aa6' AS md_ddb")
con_md_share.sql("USE md_ddb;")

# Learn about sharing here: https://motherduck.com/docs/key-tasks/querying-a-shared-motherduck-database.

In [ ]:
con_md_share.sql("SELECT * FROM Experiments")

In [ ]:
con_md_share.close()